# POPSIGN — Landmark Extraction Driver

**What this notebook does.** Drives MediaPipe Holistic landmark extraction over
the raw POPSIGN videos (~870GB, ~64K train+test clips) using
`modules/dataset/landmark/extraction.py`. **Pipeline stage 0 for POPSIGN** —
everything downstream (feature caches, training) consumes its npz output.
Replaces the deleted `popsign.0.dataset.ipynb` stub; `popsign.1.mediapipe.ipynb`
is stale and slated for retirement (TODO §0.1).

## Workflow (TODO §2)

1. **Verify the video manifests** (`data/cache/popsign/dataframes/{train,test}.csv`).
2. **Pilot batch — max 100 videos**: sweep worker counts under the resource cap,
   measure videos/s + frames/s, project full-run ETA and disk usage. **Run this
   before any bulk extraction** and pick `BEST_N_WORKERS` from its table. Pilot
   npz output is throwaway → it goes to `data/temp/popsign_pilot/` and is
   deleted by the §2 cleanup cell once `eta.json` is recorded.
3. **Full extraction** for `train` and `test` — resumable, interruption-safe.
4. **QC** — manifest stats, failure reasons, npz spot-checks.

## Output

One npz per video at `<root>/data/raw/popsign/<split>/<label>/<id>.npz`, where
`<root>` is `POPSIGN_LANDMARKS_DRIVE` from `.env` when set (else the gitignored
`src/data/raw/` tree — fine for landmarks, they are ~3 orders of magnitude
smaller than the videos). Extracted landmarks are **persistent** data, hence
`data/raw/`; only the pilot output is temp.

| key | content |
|---|---|
| `landmarks` | `(T, 543, 3)` float16, **GISLR holistic row order** (face 0-467, left_hand 468-488, pose 489-521, right_hand 522-542), NaN where undetected — `modules.dataset.landmark.subsets` indices apply unchanged |
| `fps` / `num_frames` | video metadata |

## Artifacts of this notebook (under `data/cache/popsign/extraction/`)

| path | content |
|---|---|
| `pilot_sample.json` | the seeded ≤100-video pilot sample (stable across re-runs) |
| `pilot_results.csv` | videos/s, frames/s, CPU% per worker count |
| `eta.json` | projected full-run wall time + disk usage |
| `<out_dir>/<split>/_manifest.json` | per-video extraction state (lives next to the npz output) |

## Resumability

The extraction module follows the repo manifest pattern (TODO §1.2): npz written
atomically (temp + `os.replace`) **before** the video is marked `done`; `done`
skipped, `failed` retried on re-run; manifest saves are atomic and batched
(every 50 videos — per-video rewrites of a ~30K-entry JSON would dominate
runtime; an interrupt therefore redoes at most the last <50 videos, whose
existing npz files are then *adopted* into the manifest, not re-extracted).

## Resource cap (the ≤70% rule)

- **CPU**: default worker count = `floor(0.70 × logical cores)`; each worker is
  pinned to single-threaded math libs. The pilot sweep verifies measured CPU%.
- **RAM**: the job feeder blocks while system RAM ≥ 70% (psutil backpressure).
- **GPU**: untouched — the MediaPipe GPU delegate is Ubuntu-only (README
  constraints), so extraction is CPU-only by construction.

## Design decisions vs TODO §2

- Extraction logic lives in the importable module (Windows `spawn` needs worker
  functions importable — a notebook-defined worker would break `multiprocessing`).
- `modules.paths` is imported for the tree constants only — dataset resolution
  is lazy there (`resolve_datasets()` is never called here; the manifests
  already hold absolute video paths).
- Only 1 of 4 POPSIGN train datasets is currently downloaded/enabled — the
  manifests cover that subset; regenerating with all four is TODO §2.2.

In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
from multiprocessing import cpu_count
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import modules.dataset.landmark.extraction as ex
from modules.paths import CACHE_DIR, TEMP_DIR, cleanup_temp

SEED = 42                       # pilot sampling seed
TRAIN_CSV = CACHE_DIR / "popsign" / "dataframes" / "train.csv"
TEST_CSV = CACHE_DIR / "popsign" / "dataframes" / "test.csv"
NB_CACHE_DIR = CACHE_DIR / "popsign" / "extraction"
NB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PILOT_MAX_VIDEOS = 100          # hard cap on the pilot batch
WORKER_COUNTS = (6, 10, 14, 19) # sweep; 19 = floor(0.70 × 28 logical cores)
VIDEOS_PER_TRIAL = 20           # per worker count (4 × 20 = 80 ≤ 100)
CPU_FRACTION = 0.70             # resource cap
RAM_LIMIT_PCT = 70.0

OUT_ROOT = ex.landmarks_root()  # resolved from POPSIGN_LANDMARKS_DRIVE / fallback
PILOT_ROOT = TEMP_DIR / "popsign_pilot"   # throwaway → data/temp, cleaned in §2

print(f"logical cores : {cpu_count()}  → default workers {ex.default_n_workers(CPU_FRACTION)}")
print(f"output root   : {OUT_ROOT.resolve()}")
print(f"pilot root    : {PILOT_ROOT} (temp — deleted after the pilot)")
print(f"model         : {ex.DEFAULT_MODEL_PATH}  (exists: {ex.DEFAULT_MODEL_PATH.exists()})")
print(f"nb cache      : {NB_CACHE_DIR}")

## 1. Video manifests — load & verify (TODO §2.2)

The manifests (`file_path, label, id`) were generated by an earlier iteration
from the kagglehub cache. This cell verifies they still point at real files
(kagglehub versions can shift) and summarizes them. Regeneration with all four
train datasets stays a TODO (§2.2) — three of the four downloads (~650GB) are
still commented out in `modules/paths.py`.

In [ ]:
# ============================================================
# Load manifests + spot-check that the video files exist
# ============================================================
N_SPOTCHECK = 25   # random paths verified per split

frames = {}
for split, csv in [("train", TRAIN_CSV), ("test", TEST_CSV)]:
    df = pd.read_csv(csv)
    assert {"file_path", "label", "id"} <= set(df.columns), f"{csv}: unexpected schema"
    assert df["id"].is_unique, f"{csv}: duplicate video ids"
    sample = df.sample(min(N_SPOTCHECK, len(df)), random_state=SEED)["file_path"]
    missing = [p for p in sample if not Path(p).exists()]
    frames[split] = df
    print(f"{split}: {len(df):,} videos · {df['label'].nunique()} labels · "
          f"spot-check missing {len(missing)}/{len(sample)}")
    if missing:
        print("  e.g.", missing[:3])

train_df, test_df = frames["train"], frames["test"]
train_df.groupby("label").size().describe().round(1)

## 2. Pilot batch — parameter optimization + speed (max 100 videos)

Sweeps `WORKER_COUNTS` on **disjoint** slices of a seeded 100-video sample
(disjoint so no trial is sped up by another's cached output; sample recorded to
`pilot_sample.json` so re-runs measure the same videos). Each trial reports
videos/s, frames/s and measured CPU% — pick the count with the best throughput
whose CPU stays under the cap. Pilot npz files are **throwaway** and go to the
temp tree `data/temp/popsign_pilot/w<N>/`; the cleanup cell below deletes them
once `eta.json` is recorded (the measurements in `data/cache/popsign/extraction/`
persist).

Note: the first video per worker absorbs MediaPipe graph warm-up (~seconds), so
small trials *underestimate* steady-state throughput — treat the ETA as an
upper bound.

In [ ]:
# ============================================================
# Pilot sweep — worker counts on disjoint 20-video slices
# (pilot npz -> data/temp/popsign_pilot, deleted by the cleanup cell)
# ============================================================
_pilot_file = NB_CACHE_DIR / "pilot_sample.json"
if _pilot_file.exists():
    pilot_ids = json.loads(_pilot_file.read_text())["ids"]
else:
    pilot_ids = (train_df.sample(PILOT_MAX_VIDEOS, random_state=SEED)["id"].tolist())
    _pilot_file.write_text(json.dumps(
        {"seed": SEED, "n": len(pilot_ids), "ids": pilot_ids}, indent=2))
pilot_df = train_df.set_index("id").loc[pilot_ids].reset_index()
print(f"pilot sample: {len(pilot_df)} videos "
      f"(using {len(WORKER_COUNTS) * VIDEOS_PER_TRIAL} across {len(WORKER_COUNTS)} trials)")

pilot_results = ex.benchmark_worker_counts(
    pilot_df, worker_counts=WORKER_COUNTS, videos_per_trial=VIDEOS_PER_TRIAL,
    pilot_root=PILOT_ROOT, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT)
pilot_results.to_csv(NB_CACHE_DIR / "pilot_results.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(pilot_results["n_workers"], pilot_results["videos_per_s"], "o-",
         color="tab:blue")
ax1.set_xlabel("workers")
ax1.set_ylabel("videos / s", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(pilot_results["n_workers"], pilot_results["cpu_mean_pct"], "s--",
         color="tab:red")
ax2.axhline(CPU_FRACTION * 100, color="tab:red", ls=":", lw=1)
ax2.set_ylabel("mean CPU %", color="tab:red")
ax1.set_title("Pilot: extraction throughput vs worker count (cap = 70% CPU)")
fig.tight_layout()
fig.savefig(NB_CACHE_DIR / "pilot_throughput.png", dpi=120)
plt.show()
pilot_results

In [ ]:
# ============================================================
# Pick the operating point + project full-run ETA & disk usage
# ============================================================
pilot_results = pd.read_csv(NB_CACHE_DIR / "pilot_results.csv")

# best throughput among trials that respected the CPU cap (small tolerance)
_ok = pilot_results[pilot_results["cpu_mean_pct"] <= CPU_FRACTION * 100 + 5]
best = (_ok if len(_ok) else pilot_results).sort_values(
    "videos_per_s", ascending=False).iloc[0]
BEST_N_WORKERS = int(best["n_workers"])
vps = float(best["videos_per_s"])

# disk projection from the pilot npz files (temp tree — run before cleanup;
# on a re-run after cleanup the recorded eta.json mean is reused)
_pilot_npz = list(PILOT_ROOT.rglob("*.npz")) if PILOT_ROOT.exists() else []
if _pilot_npz:
    _mean_npz_mb = float(np.mean([p.stat().st_size for p in _pilot_npz]) / 1e6)
elif (NB_CACHE_DIR / "eta.json").exists():
    _mean_npz_mb = json.loads((NB_CACHE_DIR / "eta.json").read_text())["mean_npz_mb"]
else:
    _mean_npz_mb = float("nan")

eta = {"best_n_workers": BEST_N_WORKERS, "videos_per_s": vps,
       "mean_npz_mb": round(float(_mean_npz_mb), 3)}
for split, df in [("train", pd.read_csv(TRAIN_CSV)), ("test", pd.read_csv(TEST_CSV))]:
    manifest = ex.load_manifest(OUT_ROOT / split)
    pending = len(ex.pending_jobs(df, OUT_ROOT / split, dict(manifest)))
    eta[split] = {"videos": len(df), "pending": pending,
                  "eta_h": round(pending / vps / 3600, 1),
                  "projected_gb": round(len(df) * _mean_npz_mb / 1e3, 1)}
(NB_CACHE_DIR / "eta.json").write_text(json.dumps(eta, indent=2))
print(json.dumps(eta, indent=2))

In [ ]:
# ============================================================
# Pilot cleanup — the pilot npz output is throwaway (data/temp policy):
# delete the whole temp tree now that eta.json + pilot_results.csv are
# recorded in data/cache/popsign/extraction/. Safe to re-run.
# ============================================================
assert (NB_CACHE_DIR / "eta.json").exists(), \
    "run the ETA cell first — cleanup would discard the unmeasured pilot output"
cleanup_temp()
print(f"deleted temp tree ({TEMP_DIR}) — pilot measurements persist in {NB_CACHE_DIR}")

## 3. Full extraction — train

Resumable: interrupt any time (kernel interrupt is safe — npz writes are atomic
and the manifest only ever runs behind reality, never ahead); re-running skips
everything already done. `LIMIT` allows staged runs (e.g. `LIMIT = 2000`
tonight, `None` for the rest). Progress bar shows live CPU/RAM and failure
count. Expect the `eta.json` wall time at `BEST_N_WORKERS` from §2.

In [ ]:
# ============================================================
# FULL RUN — train split (resumable; interrupt-safe)
# ============================================================
N_WORKERS = int(json.loads((NB_CACHE_DIR / "eta.json").read_text())["best_n_workers"])
LIMIT = None    # int → process only that many pending videos this run

train_summary = ex.extract_dataset(
    pd.read_csv(TRAIN_CSV), split="train", out_root=OUT_ROOT,
    n_workers=N_WORKERS, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT,
    limit=LIMIT)
print(json.dumps(train_summary, indent=2))

## 4. Full extraction — test

Identical to §3 for the test split.

In [ ]:
# ============================================================
# FULL RUN — test split (resumable; interrupt-safe)
# ============================================================
N_WORKERS = int(json.loads((NB_CACHE_DIR / "eta.json").read_text())["best_n_workers"])
LIMIT = None    # int → process only that many pending videos this run

test_summary = ex.extract_dataset(
    pd.read_csv(TEST_CSV), split="test", out_root=OUT_ROOT,
    n_workers=N_WORKERS, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT,
    limit=LIMIT)
print(json.dumps(test_summary, indent=2))

## 5. QC — manifest stats, failures, npz spot-checks

Reads only the split manifests and npz artifacts (no video access), so this is
cheap to re-run at any point mid-extraction to check progress and health.

In [ ]:
# ============================================================
# QC over both splits — progress, failures, npz spot-check
# ============================================================
N_QC_SAMPLES = 5   # npz files load-checked per split
rng = np.random.default_rng(SEED)

for split in ["train", "test"]:
    manifest = ex.load_manifest(OUT_ROOT / split)
    if not manifest:
        print(f"{split}: no manifest yet — extraction not started\n")
        continue
    st = pd.Series({k: v["status"] for k, v in manifest.items()})
    done = st[st == "done"].index
    print(f"{split}: {len(st):,} attempted — {(st == 'done').sum():,} done · "
          f"{(st == 'failed').sum():,} failed")
    fails = {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"}
    if fails:
        print("  failure reasons (top):")
        print(pd.Series(list(fails.values())).value_counts().head(5).to_string())
    nf = pd.Series({k: manifest[k].get("n_frames") for k in done}).dropna()
    if len(nf):
        print(f"  frames/video: median {nf.median():.0f} · p5 {nf.quantile(.05):.0f} "
              f"· p95 {nf.quantile(.95):.0f} · zero-frame videos {(nf == 0).sum()}")
    for vid in rng.choice(done, size=min(N_QC_SAMPLES, len(done)), replace=False):
        z = np.load(manifest[vid]["artifact"])
        lm = z["landmarks"]
        assert lm.ndim == 3 and lm.shape[1:] == (543, 3), f"bad shape {lm.shape}: {vid}"
        nan_frac = float(np.isnan(lm.astype(np.float32)).mean()) if len(lm) else 1.0
        print(f"  ok {vid}: {lm.shape} float16 · NaN {nan_frac:.1%} · fps {float(z['fps']):.0f}")
    _sz = sum(Path(manifest[k]["artifact"]).stat().st_size
              for k in done if Path(manifest[k]["artifact"]).exists())
    print(f"  on disk: {_sz / 1e9:.2f} GB\n")

## Caveats & follow-ups

- **Manifests currently cover 1 of 4 POPSIGN train datasets** — the other three
  (~650GB) are commented out in `modules/paths.py` (`resolve_datasets()`).
  Enable + download them, then regenerate the manifests and simply re-run §3
  (resumability makes the union incremental). Tracked in TODO §2.2.
- The pilot ETA is an **upper bound** (warm-up bias) but assumes the machine is
  otherwise idle; heavy concurrent use (e.g. GPU training with data loading)
  will stretch it.
- Pilot output lives in `data/temp/popsign_pilot/` and is deleted by the §2
  cleanup cell — per the repo temp policy, nothing under `data/temp/` survives
  the notebook that wrote it.
- `fps` is taken from the container metadata (`cv2.CAP_PROP_FPS`, fallback 30
  when the container reports 0) and is stored per npz; downstream consumers
  should not assume a uniform frame rate across POPSIGN.
- Landmark values are float16 (halves storage; x/y are normalized [0,1] where
  ~3 significant digits survive, which is far above landmark jitter).